# Projektaufgabe: <Titel>

**Vorlesung:** Innovative Konzepte zur Programmierung von Industrierobotern  
**Dozent:** Prof. Dr.-Ing. Björn Hein  
**Gruppe:** Alisa Hummel, Loretta Jacobs, Ole Hocker  
**Aufgabe:** 03- RRT mit kantenbewusster Erweiterung  
**Abgabedatum:** 30.07.2026  
**Vortragsdatum:** 31.07.2026

## 1. Kurzfassung
In dieser Arbeit wird das Konzept des probalisitischen Bahnplanungsverfahren "Rapidly Growing Random Trees (RRT)" um einen kantenbewussten Ansatz erweitert.

Todo: Fassen Sie kurz Problem, Ansatz, wichtigste Ergebnisse und offene Punkte zusammen.

## 2. Einleitung und Zielsetzung
Der Grundaufbau des RRT-Verfahren kann folgendermaßen beschrieben werden:
* Parameter:
    * n = maximale Anzahl von generierten Knoten im Graphen
	* k = nach wie vielen neu erstellten Knoten eine Verbindung zum Zielknoten getestet wird
	* eta = Schrittweite
* Initialisierung: Prüfe Start- und Zielpunkt auf Kollision, füge Startknoten zu leerem Graphen hinzu
* Schleife: solange Anzahl Knoten < n :
	1) **Sampling** neuer random, kollisionsfreier Punkt p_rand
	2) **Nächste-Nachbar-Suche**: Bestimme nächsten Knoten q aus Graph zu Punkt p_rand
	3) **Lokale Erweiterung:** Gehe von Nachbarknoten q die Schrittweite eta in Richtung gesampeltem Punkt p_rand, dadurch neuer Punkt p_n
	4) **Kollisionsprüfung** der Verbindung zwischen dem Nachbarknoten q und dem neuen Punkt p_n
        * wenn Verbindung kollisionsfrei ist, dann p_n als neuen Knoten und Kante zwischen p_n und Nachbar q in Graph einfügen
* Zieltest: Teste nach k neu erstellten Knoten, ob der Zielpunkt kollisionsfrei mit dem Graphen verbunden werden kann. Wenn ja füge Zielknoten und Kante dem Graphen hinzu und gebe kürzesten Pfad zurück.
* Wenn die Schleife endet, also nach n Knoten keine Verbindung zum Zielknoten hergestellt wurde, dann wurde kein Lösungspfad gefunden. Das bedeutet, das RRT-Verfahren findet nicht zwingend eine Lösung, auch wenn es theoretisch einen Pfad zwischen Start und Ziel geben würde. 

Ein knotenbasierter RRT Planer nutzt für die Nächste-Nachbar-Suche eines Punktes zum Graphen lediglich die existierenden Knoten des Graphen. Wir wollen zunächst einen erweiterten, kantenbewussten RRT Planer entwickeln indem auch die Kanten des Graphen in die Nächste-Nachbar-Suche einbezogen werden. Damit kann der Nächste-Nachbar eines Punktes entweder ein vorhandener Knoten oder die Projektion des Punktes auf eine Kante des Graphen sein. Weiter wollen wir testen, wie der kantenbewusste Ansatz das Ergebnis der Pfadplanung hinsichtlich Erfolgsrate, Pfadlänge und Planungszeit verändert. Dafür testen wir das Verfahren mit verschiedenen Parameterkombinationen und in unterschiedlichen Umgebungen mit einem 2-DoF-Punktroboter und mehreren n-DoF-Planarrobotern. Die Ergebnisse vergleichen wir mit dem knotenbewussten RRT Planer und einem Bidirektionalen-RRT Planer.

Unsere Erwartungen an den kantenbewussten Ansatz im Vergleich zum knotenbewussten Ansatz sind:
1. Kürzere Lösungspfade, denn neue Kanten zwischen hinzugefügten Knoten und dem Graphen sind immer so kurz wie möglich.
2. Findet wahrscheinlicher eine Lösung, weil neu hinzugefügte Kanten kürzer sind und damit eher kollisionsfrei.
3. Längere Planungszeit, weil die Nächste-Nachbar-Suche aufwendiger wird und mehr Rechenaufwand fordert. 


ToDo: Beschreiben Sie die konkrete Aufgabenstellung, eigene Teilziele und experimentelle Fragestellungen.

## 3. Ausgangscode und verwendete Module

Das `IPRRT.py` Modul enthält RRT Algorithmen: RRTSimple und RRT.

Zuerst werden in den Algorithmen Start und Ziel geprüft (`_checkStartGoal(startList, goalList)`), auf Kollision und die Dimensionen werden überprüft. Dann wird so lange ein zufälliger Punkt gesampelt, bis ein kollisionsfreier Punkt gefunden wird (`_getRandomFreePosition()`). Diese zwei Funktionen können für den erweiterten kantenbasierten Algorithmus übernommen werden.

Ein KDTree wird dann benutzt um den nächsten Nachbarn zu finden (`kdTree.query(pos, k=1)`). Der KDTree kann in unserem Algorithmus bei der nächsten Nachbar Suche helfen. Dafür müssen jedoch durch die Projektion erzeugten Punkte zum Aufbau des KDTrees verwendet werden und nicht die Knoten des Graphen. Der KDTree kann auch benutzt werden um Punkte und damit die zugehörigen Kanten innerhalb eines bestimmten Radius um den Punkt p zu finden. Dabei muss bei `query(x, k=1, eps=0.0, p=2.0, distance_upper_bound=inf, workers=1)` die `distance_upper_bound` auf einen bestimmen Radius gesetzt werden. Der Punkt p wird dann auf diese Kanten projiziert um Lotfußpunkte für die Kanten zu finden, mit diesen wird die vorher erwähnte nächste Nachbar Suche gemacht.

Nachdem eine Kante gefunden wurde, die dem Graphen hinzugefügt werden soll, wird sie auf Kollision geprüft. Danach wird der neue Knoten zum Graph hinzugefügt `graph.add_node(self.lastGeneratedNodeNumber, pos=pos)` mit einer ID `lastGeneratedNodeNumber`. Die ID wird jedes mal um eins erhöht wenn ein Knoten hinzugefügt wird. Die Kante wird auch hinzugefügt `graph.add_edge(result[1], self.lastGeneratedNodeNumber)`. Dabei werden Start- und Endpunkt der Kante übergeben anhand von der ID des Knotens. Für unseren Algorithmus können wir diese Funktionen ebenfalls verwenden und vor allem sind die beim Trennen der Kante hilfreich. Dafür müssen wir aufpassen, dass wir auch die ID erhöhen wenn ein Knoten hinzugefügt wird.
    
In der `Config` Dictionary (`testGoalAfterNumberOfNodes = k`) wird festgelegt, nach wie vielen angelegten Knoten k im Graphen getestet wird ob Zielknoten verbunden werden kann (`self.lastGeneratedNodeNumber % config["testGoalAfterNumberOfNodes"]`). Das entspricht nicht zwangsläufig k Schleifendurchläufen, weil in einem Schleifendurchlauf kein Knoten angelegt werden zum Beispiel wenn eine Kollision bei der gefundenen Kante auftaucht. Bei der Umgestaltung auf unserem Algorithmus muss auch beachtet werden, dass beim Trennen von Kanten Knoten erzeugt werden und somit in einem Schleifendurchlauf 2 Knoten hinzugefügt werden können. Daher muss entweder der Wert `k` angepasst oder eine zusätzliche Variable verwendet werden.

Die zwei Algorithmen RRT und RRTSimple unterscheiden sich darin, dass RRTSimple den gesampelten Punkt und nächsten Nachbarn benutzt um die neue Kante und den neuen Knoten zu finden, während RRT eine kürzere Kante hinzufügt. Diese Kante wird gefunden indem nur eine Schrittweite in Richtung gesampelten Punktes gegangen wird (`newPos = 0.5 * (end - start) + start`) und dann die Kante zwischen dem nähestem Nachbarn `result` und `newPos` genommen wird.

In den Algorithmen RRT und RRTSimple spielen Kanten bei der Auswahl des Erweiterungspunkts bisher keine Rolle, da der KDTree nur aus den Punkten im Graph erstellt wird. 

## 4. Konzept und Algorithmus

ToDo: Beschreiben Sie den Algorithmus fachlich. Nutzen Sie Skizzen, Pseudocode, Tabellen oder kurze Formeln, wenn sie helfen.

### Projektion Punkt auf Kante
Wir berechnen den Lotfußpunkt des Punktes auf der Kante. 

* Gegeben: 
    * Kante mit Startpunkt $q_a$ und Endpunkt $q_b$
    * Punkt $p$ der projiziert werden soll
* Gesucht: 
    * projizierter Punkt $p'$
    * Parameter $t$, der die Position von $p'$ auf der Geraden durch $q_a$ und $q_b$ beschreibt
* Mathematik:
    * Vektor $AB = q_b - q_a$
    * Vektor $AP = p - q_a$
    * $t = \frac{\langle {AB,AP} \rangle}{\langle {AB,AB} \rangle}$ ; wobei $\langle {\cdot,\cdot} \rangle$ das Skalarprodukt der beiden Vektoren berechnet

Die Projektion des Punktes muss auf das Kantensegment beschränkt werden, also zwischen Start- und Endpunkt der Kante liegen ($t \in [0, 1]$). In Fällen, in denen der Lotfußpunkt außerhalb der Kante liegt, wird der projizierte Punkt der nächstgelegene Endpunkt der Kante.
* Ergebnis
    * $t<0 \Rightarrow p' = q_a$
    * $t>1 \Rightarrow p' = q_b$
    * $t \in [0, 1] \Rightarrow p' = q_a + t*q_b$

Die beschriebene Projektion ist in der Methode `PointProjection.projectPointOnEdge(p, q_a, q_b)` implementiert.


## Kante Aufteilen
Aufteilen einer Kante in zwei Teile
* Gegeben: 
    * Kante mit Startpunkt $q_{start}$ und Endpunkt $q_{end}$
    * Punkt $p$ der die Kante aufteilt
* Vorgehen:
    * Kante ($q_{start}$, $q_{end}$) entfernen
    * Punkt $p$ zum graph hinzufügen
    * Kanten ($q_{start}$, $p$) ($p$, $q_{end}$) hinzufügen

Der Graph soll nach der Funktion immernoch verbunden und Kreisfrei sein. Da der Graph nicht mit Kantengewichten arbeitet haben wir nicht die Kantengewichte übernehmen oder neuberechnen müssen. 
Nach der Funktion ist keine neue Kollisionsprüfung nötig, da der gegebener Punkt auf der Kante liegt und die 2 neu gestalteten Kanten auf der ehemaligen Kante liegen. Der einzige Grund für eine Kollision wäre, wenn es bei der Kollisionsprüfung von der ehemaligen Kante auch eine Kollision gab und übersehen wurde.

## 5. Implementierung

Erläutern Sie die wichtigsten Implementierungsentscheidungen. Umfangreicher Code soll in Modulen liegen und hier importiert werden.

In [ ]:
# Eigene Module importieren
# Beispiel:
# from my_planner import MyPlanner
from BiRRT import BiRRT
from lib.IPRRT import RRT
pass

## 6. Validierung an kleinen Beispielen

Zeigen Sie an kleinen, nachvollziehbaren Fällen, dass die zentralen Bausteine korrekt funktionieren.

In [ ]:
# Kleine Tests oder Plausibilitaetschecks ausfuehren.
pass

## 7. Experimente und Benchmarks

Beschreiben Sie Testumgebungen, Parameter, Metriken und Anzahl der Wiederholungen.

In [ ]:
# Benchmark-Konfigurationen definieren.
benchmarks = []
configs = []

import itertools
from typing import Any, Dict, List, Tuple, TypedDict

import numpy as np
from pandas import DataFrame

from joblib import Parallel, delayed
from lib.IPVISRRT import rrtPRMVisualize
from lib.IPPerfMonitor import IPPerfMonitor
from lib.IPBenchmark import Benchmark
import lib.IPTestSuite as ts
import matplotlib.pyplot as plt
import tqdm
import networkx as nx


class TestConfig(TypedDict):
    numberOfGeneratedNodes: int
    balanceTree: bool
    stepSize: float


sweepNumberOfGeneratedNodes = [10, 20, 50, 100, 150]
sweepStepSize = np.linspace(0.5, 22, 44)
sweepBalanceTree = [True, False]

numTestCases = len(sweepNumberOfGeneratedNodes) * len(sweepStepSize) * len(sweepBalanceTree)

testCases1: Dict[Tuple[int, int], Tuple[str, TestConfig]] = {}
metrics1: Dict[Tuple[int, int], Dict[str, Any]] = {}
timings1: Dict[Tuple[int, int], DataFrame] = {}

def _solution_length(solution, graph):
    """Berechne euklidische Pfadlänge einer Lösung."""
    if solution is None or solution == []:
        return np.nan
    try:
        subgraph = nx.subgraph(graph, solution)
        positions = nx.get_node_attributes(subgraph, 'pos')
        if not positions:
            return np.nan
        segment_lengths = np.linalg.norm(np.diff(list(positions.values()), axis=0), axis=1)
        return float(np.sum(segment_lengths))
    except TypeError:
        return np.nan

def _run_benchmark_case(
    benchmark: Benchmark,
    bench_rep: int,
    test_index: int,
    stepSize: float,
    balanceTree: bool,
    numGenNodes: int,
 ) -> Dict[str, Any]:
    IPPerfMonitor.clearData()

    currentConfig = TestConfig(
        stepSize=stepSize,
        balanceTree=balanceTree,
        numberOfGeneratedNodes=numGenNodes,
    )

    rrt = BiRRT(benchmark.collisionChecker)
    solution = rrt.planPath(benchmark.startList, benchmark.goalList, currentConfig)
    timings = IPPerfMonitor.dataFrame()

    return {
        "key": (test_index, bench_rep),
        "benchmark_name": benchmark.name,
        "config": currentConfig,
        "solution": solution,
        "graph": rrt.graph,
        "timings": timings,
    }

benchmark_tasks = []
for benchRep, benchmark in enumerate(ts.benchList * 100):
    for testIndex, (stepSize, balanceTree, numGenNodes) in enumerate(
        itertools.product(
            sweepStepSize,
            sweepBalanceTree,
            sweepNumberOfGeneratedNodes,
        )
    ):
        benchmark_tasks.append(
            delayed(_run_benchmark_case)(
                benchmark,
                benchRep,
                testIndex,
                stepSize,
                balanceTree,
                numGenNodes,
            )
        )

benchmark_results = Parallel(n_jobs=16, backend="loky")(
    tqdm.tqdm(
        benchmark_tasks,
        total=len(benchmark_tasks),
        desc="Benchmark",
    )
)

for result in benchmark_results:
    key = result["key"]
    testCases1[key] = (result["benchmark_name"], result["config"])
    metrics1[key] = {
        "solution_len": _solution_length(result["solution"], result["graph"]),
        "num_nodes": len(result["graph"].nodes()),
        "num_edges": len(result["graph"].edges()),
    }
    #timings1[key] = result["timings"]

# Hypothese: Wird die bottleneck-performance verbessert? Was ist die Wahrscheinlichkeit, dass eine Kante orthogonal zum Bottleneck liegt?
# Hypothese: Kann man die Performance verbessern, indem man projektionspunkte nicht neu erzeugt, wenn sie nicht weit von den Kantenenden entfernt sind?

In [ ]:
# plot average solution length per benchmark and config, split by balanceTree

plot_rows = []
for key, metrics in metrics1.items():
    benchmark_name, config = testCases1[key]
    plot_rows.append(
        {
            "benchmark": benchmark_name,
            "benchRep": key[1],
            "testIndex": key[0],
            "stepSize": config["stepSize"],
            "numberOfGeneratedNodes": config["numberOfGeneratedNodes"],
            "balanceTree": config["balanceTree"],
            "solution_len": metrics["solution_len"],
        }
    )

plot_df = DataFrame(plot_rows).copy()
success_df = plot_df.dropna(subset=["solution_len"]).copy()

if success_df.empty:
    print("No benchmark data available for plotting.")
else:
    summary_df = (
        success_df.groupby(
            ["benchmark", "numberOfGeneratedNodes", "balanceTree", "stepSize"],
            dropna=False,
        )
        .agg(
            mean_solution_len=("solution_len", "mean"),
            std_solution_len=("solution_len", "std"),
            runs=("solution_len", "size"),
        )
        .reset_index()
        .sort_values(["benchmark", "numberOfGeneratedNodes", "balanceTree", "stepSize"])
    )

    benchmark_names = summary_df["benchmark"].drop_duplicates().tolist()
    balance_values = [True, False]
    nrows = len(benchmark_names)
    ncols = len(balance_values)

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(7 * ncols, 6.8 * nrows),
        sharex=True,
        sharey=False,
    )
    axes = np.atleast_2d(axes)

    for row_index, benchmark_name in enumerate(benchmark_names):
        subset = summary_df[summary_df["benchmark"] == benchmark_name]

        for col_index, balance_value in enumerate(balance_values):
            ax = axes[row_index, col_index]
            balance_subset = subset[subset["balanceTree"] == balance_value]
            balance_label = "balanced" if balance_value else "unbalanced"
            table_rows = []

            for num_gen_nodes, group in balance_subset.groupby("numberOfGeneratedNodes"):
                group = group.sort_values("stepSize")
                errorbar = ax.errorbar(
                    group["stepSize"],
                    group["mean_solution_len"],
                    yerr=group["std_solution_len"],
                    marker="o",
                    linewidth=1.8,
                    capsize=3,
                    label=f"n={num_gen_nodes}",
                )
                curve_color = errorbar[0].get_color()

                min_index = group["mean_solution_len"].idxmin()
                min_row = group.loc[min_index]
                ax.scatter(
                    [min_row["stepSize"]],
                    [min_row["mean_solution_len"]],
                    marker="*",
                    s=200,
                    facecolor=curve_color,
                    edgecolor="black",
                    linewidth=1.2,
                    zorder=6,
                )
                ax.annotate(
                    f"{min_row['stepSize']:.1f}",
                    xy=(min_row["stepSize"], min_row["mean_solution_len"]),
                    xytext=(0, 10),
                    textcoords="offset points",
                    ha="center",
                    va="bottom",
                    fontsize=8,
                    color=curve_color,
                    bbox={"boxstyle": "round,pad=0.15", "facecolor": "white", "edgecolor": curve_color, "alpha": 0.85},
                )

                raw_values = plot_df[
                    (plot_df["benchmark"] == benchmark_name)
                    & (plot_df["balanceTree"] == balance_value)
                    & (plot_df["numberOfGeneratedNodes"] == num_gen_nodes)
                ]["solution_len"]
                successful_values = raw_values.dropna()
                found_paths = int(successful_values.size)
                total_runs = int(raw_values.size)
                success_rate = found_paths / total_runs if total_runs > 0 else 0.0

                table_rows.append(
                    [
                        f"n={num_gen_nodes}",
                        f"{found_paths}/{total_runs}",
                        f"{success_rate:.2%}",
                        f"{successful_values.mean():.2f}" if found_paths > 0 else "-",
                        f"{successful_values.std():.2f}" if found_paths > 1 else "-",
                    ]
                )

            ax.set_title(f"{benchmark_name} | {balance_label}")
            ax.set_xlabel("stepSize")
            ax.set_ylabel("avg. solution path length")
            ax.set_yticks(np.arange(0, summary_df["mean_solution_len"].max() + 5, 50))
            ax.grid(True, alpha=0.25)
            ax.legend(fontsize=8)
            ax.set_anchor("N")
            ax.set_xlim(summary_df["stepSize"].min() - 0.25, summary_df["stepSize"].max() + 0.25)
            x_ticks = np.arange(summary_df["stepSize"].min(), summary_df["stepSize"].max() + 0.5, 0.5)
            ax.set_xticks(x_ticks)
            ax.set_xticklabels([f"{tick:.1f}" for tick in x_ticks])
            ax.tick_params(axis="x", which="major", labelrotation=45)
            ax.tick_params(axis="x", which="minor", length=3)

            if table_rows:
                table = ax.table(
                    cellText=table_rows,
                    colLabels=["config", "found", "rate", "avg", "std"],
                    cellLoc="center",
                    colLoc="center",
                    bbox=[0.0, -0.56, 1.0, 0.34],
                )
                table.auto_set_font_size(False)
                table.set_fontsize(7)
                table.scale(1.0, 1.05)

    fig.suptitle("Average solution length per benchmark, configuration, and balance setting", y=1.02)
    plt.tight_layout(rect=[0, 0.03, 1, 0.98])
    fig

# Test 2

In [ ]:

solutions2 = []

IPPerfMonitor.clearData()
for benchmark in ts.benchList * 100:
    try:
        rrt = RRT(benchmark.collisionChecker)
        solution = rrt.planPath(benchmark.startList, benchmark.goalList, rrtConfig)

        solutions2.append(solution)
        if False:
            fig_local = plt.figure(figsize=(10,10))
            ax = fig_local.add_subplot(1,1,1)
            title = benchmark.name
            if solution == []:
                title += " (No path found!)"
            title += "\n Assumed complexity level " + str(benchmark.level)
            
            ax.set_title(title)
            rrtPRMVisualize(rrt, solution, ax=ax, nodeSize=50)
        
    except Exception as e:
        print("ERROR: ",benchmark.name, e)
df = IPPerfMonitor.dataFrame()

summary2 = (
    df.groupby("name", dropna=False)
        .agg(
            calls=("name", "size"),
            avg_time=("time", "mean"),
            total_time=("time", "sum"),
        ).sort_values("calls", ascending=False)
)


np.average([len(solution) for solution in solutions2 if solution != []])

In [ ]:
np.count_nonzero([len(solution) for solution in solutions2])

In [ ]:
summary2

## 8. Visualisierungen und Animationen

Zeigen Sie Suchraum, Roadmap/Baum, Pfad, Kollisionen, Statistiken oder Animationen.

In [ ]:
# Visualisierungen erzeugen.
pass

## 9. Ergebnisse

Stellen Sie Ergebnisse in Tabellen und Diagrammen dar und erklären Sie beobachtete Effekte.

In [ ]:
# Ergebnisse als DataFrame/Tabelle/Plot darstellen.
pass

## 10. Diskussion

Diskutieren Sie, was funktioniert hat, wo Grenzen liegen, welche Parameter wichtig sind und wie belastbar die Ergebnisse sind.

## 11. Fazit

Fassen Sie die wichtigsten Erkenntnisse knapp zusammen.

## 12. Verwendung von KI-Werkzeugen

Dokumentieren Sie, wofür KI verwendet wurde, welche Vorschläge übernommen oder verworfen wurden und wie die Korrektheit geprüft wurde.

## 13. Präsentationsnotizen

Notieren Sie die Kernaussagen für die Präsentation: Problem, Ansatz, wichtigste Visualisierung, wichtigste Ergebnisse und wichtigste Erkenntnis.